In [3]:
import pickle

# Load embedding
# review collaborative embeddings: "../../data/games_with_embeddings.pkl"
# game description embeddings: "../../data/game_descriptions_embeddings.pkl"
with open("../../data/game_descriptions_embeddings.pkl", "rb") as f:
    game_embedding_dict = pickle.load(f)

In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array (size 32)
    :param top_n: Number of recommendations to return
    """
    
    # Check if the game exists in our database
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    # Isolate the target vector and reshape it for sklearn
    # reshape(1, -1) turns it from shape (32,) to (1, 32)
    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    # Prepare the rest of the data
    # We separate names and vectors so their indexes match up
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    # Calculate Cosine Similarity
    # This compares the target (1, 32) against all games (N, 32) simultaneously
    # It returns an array of scores from -1.0 (opposites) to 1.0 (identical)
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    
    # Sort the results
    # argsort() gives us the indexes from lowest to highest, so we reverse it [::-1]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    # Format the output
    print(f"Games most similar to '{target_game}':\n")
    results = []
    
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        
        # Skip the target game itself (it will always have a 1.0 score)
        if match_name == target_game:
            continue
            
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        
        # Stop once we hit our desired number of recommendations
        if len(results) == top_n:
            break
            
    return results

In [15]:
# For reproducible random numbers
np.random.seed(42) 

# Run the function
find_closest_games("Marvel's Spider-Man: Miles Morales", game_embedding_dict, top_n=10)

Games most similar to 'Marvel's Spider-Man: Miles Morales':

1. Marvel's Spider-Man 2 (Similarity Score: 0.7476)
2. Marvel's Spider-Man Remastered (Similarity Score: 0.6935)
3. Marvel's Spider-Man - Silver Lining (Similarity Score: 0.6237)
4. Spider-Man: The Movie (Similarity Score: 0.6008)
5. Spider-Man: Web of Shadows (Similarity Score: 0.5960)
6. Spider-Man: Battle for New York (Similarity Score: 0.5661)
7. Spider-Man 3 (Similarity Score: 0.5627)
8. The Amazing Spider-Man vs. the Kingpin (Similarity Score: 0.5578)
9. Spider-Man Unlimited (Similarity Score: 0.5468)
10. Spider-Man 2: The Sinister Six (Similarity Score: 0.5452)


[("Marvel's Spider-Man 2", np.float32(0.74763364)),
 ("Marvel's Spider-Man Remastered", np.float32(0.6934806)),
 ("Marvel's Spider-Man - Silver Lining", np.float32(0.62365746)),
 ('Spider-Man: The Movie', np.float32(0.60083675)),
 ('Spider-Man: Web of Shadows', np.float32(0.59602416)),
 ('Spider-Man: Battle for New York', np.float32(0.5660621)),
 ('Spider-Man 3', np.float32(0.56270444)),
 ('The Amazing Spider-Man vs. the Kingpin', np.float32(0.55783224)),
 ('Spider-Man Unlimited', np.float32(0.5468297)),
 ('Spider-Man 2: The Sinister Six', np.float32(0.5452018))]